# Biohub - Cell Tracking Baseline Pipeline

This notebook is a baseline pipeline for creating submissions for the **Biohub - Cell Tracking During Development** competition.
このノートブックは、**Biohub - Cell Tracking During Development** コンペティション向けのベースライン提出を作成するパイプラインです。

It performs the following steps:
以下の手順を実行します。

1. Load test Zarr data
1. テスト用Zarrデータの読み込み
2. 3D cell detection using `scikit-image`'s `blob_dog`
2. `scikit-image` の `blob_dog` を用いた3D細胞検出
3. Cell tracking (edge creation) between time frames using nearest neighbor search
3. 最近傍探索によるタイムフレーム間の細胞トラッキング(エッジ作成)
4. Create and save `submission.csv` according to the specified schema
4. 指定のスキーマに沿った `submission.csv` の作成と保存

In [ ]:
# Install additional libraries in Kaggle offline environment.
# Kaggle環境(インターネットOFF)でプリセット外ライブラリをインストール。
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr

In [ ]:
import os
import glob
import zarr
import numpy as np
import pandas as pd
from skimage.feature import blob_dog
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from tqdm import tqdm

## 1. Setup and Data Path Verification
## 1. 設定とデータパスの確認

Set up the input data paths in the Kaggle environment.
Kaggle環境でのテスト用データセットのパスを設定します。

In [ ]:
# Candidate input data paths in Kaggle
# Kaggleでの入力データパスの候補
CANDIDATES = [
    "/kaggle/input/biohub-cell-tracking-during-development",
    "/kaggle/input/competitions/biohub-cell-tracking-during-development",
]

ROOT = "/kaggle/input/biohub-cell-tracking-during-development"
for p in CANDIDATES:
    if os.path.exists(os.path.join(p, "test")):
        ROOT = p
        break

TEST_DIR = os.path.join(ROOT, "test")
print(f"Using TEST_DIR: {TEST_DIR}")

# Get the list of Zarr folders in the test dataset
# テストデータセットのZarrフォルダ一覧を取得
test_zarr_paths = glob.glob(os.path.join(TEST_DIR, "*.zarr"))
print(f"Found {len(test_zarr_paths)} test datasets.")
for p in test_zarr_paths:
    print(f"  {os.path.basename(p)}")

## 2. 3D Blob Detection and Tracking Implementation
## 2. 3Dブロブ検出とトラッキングの実装

Detect cells in each time frame and track them between adjacent frames.
各タイムフレームで細胞を検出し、隣接フレーム間でトラッキングを行います。

In [ ]:
def detect_cells_3d(image_3d, min_sigma=2, max_sigma=5, threshold=0.05):
    """Detect cell centroids (Z,Y,X) from a 3D image.
    3D画像から細胞の重心(Z,Y,X)を検出します。
    """
    # Min-Max normalize the image to 0.0-1.0 to maximize contrast
    # 画像を 0.0〜1.0 にMin-Max規格化してコントラストを最大化します
    img_min = image_3d.min()
    img_max = image_3d.max()
    if img_max > img_min:
        img_norm = (image_3d.astype(np.float32) - img_min) / (img_max - img_min)
    else:
        img_norm = np.zeros_like(image_3d, dtype=np.float32)
        
    # Run blob_dog on the normalized image
    # 規格化画像に対して blob_dog を実行
    blobs = blob_dog(img_norm, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
    # Since blobs returns an array of (z, y, x, sigma), extract coordinates only
    # blobsは (z, y, x, sigma) の配列を返すため、座標のみ抽出
    if len(blobs) > 0:
        return blobs[:, :3]
    return np.empty((0, 3))

In [ ]:
def track_frame_to_frame(coords_prev, coords_curr, max_distance=15.0):
    """Perform nearest neighbor matching between adjacent frames.
    隣接するフレーム間で最近傍マッチングを行います。
    Returns: List of (prev_index, curr_index) pairs
    戻り値: (prev_index, curr_index) のペアリスト
    """
    if len(coords_prev) == 0 or len(coords_curr) == 0:
        return []
    
    # Compute distance matrix
    # 距離行列を計算
    dists = cdist(coords_prev, coords_curr)
    
    links = []
    # Simple nearest neighbor matching using a greedy method
    # シンプルな貪欲法(Greedy)による最近傍マッチング
    # For more robust matching, algorithms like the Hungarian method can be used
    # より堅牢にするにはハンガリアン法等を使用できます
    used_curr = set()
    for i in range(len(coords_prev)):
        js = np.argsort(dists[i])
        for j in js:
            if j not in used_curr and dists[i, j] <= max_distance:
                links.append((i, j))
                used_curr.add(j)
                break
    return links

## 3. Pipeline Execution
## 3. パイプラインの実行

Loop through all test datasets to collect node and edge information.
すべてのテストデータセットに対して処理をループ実行し、ノードとエッジの情報を収集します。

In [ ]:
# Determine operation mode (if file count <= 2, treat as development/test environment)
# 1. 動作モードの判定 (ファイル数が2以下なら開発・テスト環境とみなす)
IS_DEVELOPMENT = (len(test_zarr_paths) <= 2)

nodes = []
edges = []

if len(test_zarr_paths) == 0:
    print("Warning: No test datasets found. Running in interactive mock mode to allow 'Run All' to pass.")
    # Fallback when the editor screen is completely empty
    # エディタ編集画面で完全に空の場合のフォールバック
    nodes.append({
        "dataset": "mock_dataset",
        "row_type": "node",
        "node_id": 1,
        "t": 0,
        "z": 10,
        "y": 10,
        "x": 10,
        "source_id": -1,
        "target_id": -1
    })
else:
    print(f"Running in {'development' if IS_DEVELOPMENT else 'production'} mode. Processing {len(test_zarr_paths)} datasets.")
    for zarr_path in test_zarr_paths:
        dataset_name = os.path.basename(zarr_path).replace(".zarr", "")
        print(f"Processing dataset: {dataset_name}")
        
        # Reset node ID from 1 for each dataset
        # 各データセットごとにノードIDを1からリセット
        node_counter = 1
        
        # Open Zarr store
        # Zarrストアを開く
        store = zarr.open(zarr_path, mode='r')
        # Get the highest resolution array "0"
        # 最高解像度の配列 "0" を取得
        arr = store['0']
        
        # Check shape (T, C, Z, Y, X) or (T, Z, Y, X)
        # 形状の確認 (T, C, Z, Y, X) または (T, Z, Y, X)
        has_channels = (arr.ndim == 5)
        num_frames = arr.shape[0]
        
        prev_nodes_info = []  # Node info of the previous frame / 前フレームのノード情報
        
        for t in tqdm(range(num_frames), desc="Frames"):
            # Slice 3D image
            # 3D画像の切り出し
            if has_channels:
                img_3d = arr[t, 0, :, :, :]
            else:
                img_3d = arr[t, :, :, :]
                
            # 3D cell detection
            # 3D細胞検出
            coords = detect_cells_3d(img_3d, min_sigma=2, max_sigma=5, threshold=0.05)
            
            # [Warning Log] When detected cells count is 0 (can happen in actual microscope data due to out-of-field or dark environments)
            # 【警告ログ】細胞検出数が0個の場合 (実際の顕微鏡データでは視野外や暗室などの理由で0件になり得ます)
            if len(coords) == 0:
                min_val = np.min(img_3d)
                max_val = np.max(img_3d)
                mean_val = np.mean(img_3d)
                print(f"Warning: 0 cells detected in dataset '{dataset_name}' at frame {t}. "
                      f"Image Stats -> Min: {min_val}, Max: {max_val}, Mean: {mean_val:.4f}")
            
            curr_nodes_info = []
            # Create node rows
            # ノード行の作成
            for idx, (z, y, x) in enumerate(coords):
                node_id = node_counter
                node_counter += 1
                
                nodes.append({
                    "dataset": dataset_name,
                    "row_type": "node",
                    "node_id": int(node_id),
                    "t": int(t),
                    "z": int(round(z)),
                    "y": int(round(y)),
                    "x": int(round(x)),
                    "source_id": -1,
                    "target_id": -1
                })
                curr_nodes_info.append((idx, node_id, (z, y, x)))
                
            # Create edge rows (tracking)
            # エッジ行の作成(トラッキング)
            if len(prev_nodes_info) > 0 and len(curr_nodes_info) > 0:
                coords_prev = np.array([item[2] for item in prev_nodes_info])
                coords_curr = np.array([item[2] for item in curr_nodes_info])
                
                # Convert to physical scale (micrometers) and compute tracking distance
                # 物理スケール(µm)に変換してトラッキング距離を計算 (Z: 1.625, Y: 0.40625, X: 0.40625)
                scale_zyx = np.array([1.625, 0.40625, 0.40625])
                coords_prev_physical = coords_prev * scale_zyx
                coords_curr_physical = coords_curr * scale_zyx
                
                links = track_frame_to_frame(coords_prev_physical, coords_curr_physical, max_distance=15.0)
                
                # [Warning Log] When connected edges count is 0 (can happen due to sudden cell movement or microscope drift)
                # 【警告ログ】接続が0個の場合 (細胞の急激な移動や顕微鏡のドリフトで0件になり得ます)
                if len(links) == 0:
                    print(f"Warning: 0 tracking edges created between frame {t-1} and {t} in dataset '{dataset_name}'. "
                          f"Previous node count: {len(prev_nodes_info)}, Current node count: {len(curr_nodes_info)}.")
                
                for idx_prev, idx_curr in links:
                    src_id = prev_nodes_info[idx_prev][1]
                    tgt_id = curr_nodes_info[idx_curr][1]
                    
                    edges.append({
                        "dataset": dataset_name,
                        "row_type": "edge",
                        "node_id": -1,
                        "t": -1,
                        "z": -1,
                        "y": -1,
                        "x": -1,
                        "source_id": int(src_id),
                        "target_id": int(tgt_id)
                    })
                    
            prev_nodes_info = curr_nodes_info

## 4. Submission File Generation and Verification
## 4. 提出ファイルの生成と確認

Merge nodes and edges and output to CSV in the specified format.
ノードとエッジをマージして所定のフォーマットでCSVに出力します。

In [ ]:
# Convert to DataFrame
# DataFrameに変換
columns_order = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]

if len(nodes) == 0:
    df_nodes = pd.DataFrame(columns=columns_order)
else:
    df_nodes = pd.DataFrame(nodes)

if len(edges) == 0:
    df_edges = pd.DataFrame(columns=columns_order)
else:
    df_edges = pd.DataFrame(edges)

df_sub = pd.concat([df_nodes, df_edges], ignore_index=True)

# Add id column (sequential index) at the beginning
# id列(連番インデックス)を最初に追加
df_sub.insert(0, 'id', range(len(df_sub)))

# Reorder columns and check data types
# カラム順序の整理とデータ型の確認
df_sub = df_sub[["id"] + columns_order]

# Explicitly cast data types (convert all to int)
# データ型を明示的にキャスト (全てをint型にする)
df_sub = df_sub.astype({
    'id': 'int64',
    'node_id': 'int64',
    't': 'int64',
    'z': 'int64',
    'y': 'int64',
    'x': 'int64',
    'source_id': 'int64',
    'target_id': 'int64'
})

# Preview display
# プレビュー表示
print(f"Total rows: {len(df_sub)}")
print(df_sub.head())
print(df_sub.tail())

# Save
# 保存
df_sub.to_csv("submission.csv", index=False)
print("submission.csv has been successfully generated!")

## 5. How to Submit to Kaggle
## 5. Kaggleへの提出方法

Commit & submit on Kaggle Notebook, or submit using the Kaggle CLI command below:
Kaggle Notebook上でのコミット & 提出、または以下のKaggle CLIコマンドを使用して提出します。
```bash
# kaggle competitions submit -c biohub-cell-tracking-during-development -f submission.csv -m "Baseline dog detector + nearest neighbor tracker"
```